# Failure analysis

In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC

from src.dataloader import load_all
from src.evaluation import author_codes
from src.features import download_resources, extract_all
from src.model import Classifier

download_resources()
splits = load_all()
train_texts, train_y = splits["train"]
dev_texts, dev_y = splits["dev"]
test_texts, test_y = splits["test"]
train_y = np.array(train_y)
test_y = np.array(test_y)

best = pd.read_csv("outputs/tuning.csv").sort_values("macro_f1").iloc[-1]
best

classifier    linear SVM
C               0.031623
micro_f1        0.966418
macro_f1        0.953372
Name: 3, dtype: object

In [2]:
train_X = extract_all(train_texts)
dev_X = extract_all(dev_texts)
test_X = extract_all(test_texts)

model = Classifier(LinearSVC(C=best["C"], max_iter=20000)).train(train_X + dev_X, list(train_y) + dev_y).pipe

scores = model.decision_function(test_X)
labels = model.classes_
pred = labels[scores.argmax(axis=1)]
wrong = pred != test_y
codes = author_codes(labels)
print(wrong.sum(), "errors out of", len(test_y))

14 errors out of 267


## How certain are we for Errors

In [3]:
top2 = np.sort(scores, axis=1)[:, -2:]
margin = top2[:, 1] - top2[:, 0]
true_rank = (scores > scores[np.arange(len(test_y)), np.searchsorted(labels, test_y)][:, None]).sum(axis=1) + 1

print("median margin, correct:", round(np.median(margin[~wrong]), 2))
print("median margin, wrong:  ", round(np.median(margin[wrong]), 2))
print("errors with the true author in the top 3:", round(np.mean(true_rank[wrong] <= 3), 2))

median margin, correct: 1.25
median margin, wrong:   0.3
errors with the true author in the top 3: 0.71


## Are errors spread out

In [4]:
per_author = pd.DataFrame({
    "author": [codes[a] for a in labels],
    "n_test": [np.sum(test_y == a) for a in labels],
    "f1": f1_score(test_y, pred, labels=labels, average=None),
})
per_author.sort_values("f1")

,author,n_test,f1
17,R,8,0.857143
10,K,18,0.882353
8,I,12,0.888889
15,P,10,0.909091
12,M,12,0.916667
16,Q,7,0.923077
1,B,15,0.928571
7,H,15,0.933333
19,T,7,0.933333
3,D,17,0.937500


## Each error

Every misclassified test snippet; `i` is its index in the test set, so the full text is `test_texts[i]`.

In [5]:
errors = pd.DataFrame({
    "i": np.flatnonzero(wrong),
    "true": [codes[a] for a in test_y[wrong]],
    "pred": [codes[a] for a in pred[wrong]],
    "margin": margin[wrong].round(2),
    "true_rank": true_rank[wrong],
    "start": [test_texts[i][:80] for i in np.flatnonzero(wrong)],
})
errors.sort_values("margin")

,i,true,pred,margin,true_rank,start
4,95,D,I,0.04,2,"Konichiwa! Firs of all, arigatou gozaimasu for..."
0,31,M,G,0.06,2,"""Anyway,"" Edward continued, ""I was out buying ..."
7,153,R,I,0.09,2,He was smart enough to know that in all likeli...
3,94,B,K,0.10,2,How could I not spot it right away? It is actu...
1,41,R,O,0.13,2,The old woman placed her free hand on top of S...
2,78,O,T,0.22,2,"""Do you really think you are the end all for m..."
9,168,K,M,0.29,2,"But contrary to popular belief, I was doing it..."
5,113,Q,C,0.31,6,"Mary had even smiled at her as she passed, and..."
13,238,F,H,0.35,2,"Every time I had to fight against Kuja, I knew..."
10,177,H,I,0.52,5,"Then skin grew outwards, and before long this ..."


In [6]:
# each error in full, next to the closest (in scaled feature space) train + dev snippet of the predicted and of the true author
fit_texts = train_texts + dev_texts
fit_y = np.array(list(train_y) + dev_y)
fit_Z = model["scaler"].transform(train_X + dev_X)
test_Z = model["scaler"].transform(test_X)

for i in errors.sort_values("margin")["i"]:
    t, p = test_y[i], pred[i]
    print(f"test {i}: true {codes[t]}, predicted {codes[p]}, margin {margin[i]:.2f}")
    print(test_texts[i])
    print()


test 95: true D, predicted I, margin 0.04
Konichiwa! Firs of all, arigatou gozaimasu for all those who reviewed! You inspired me a lot. Thanks also for the corrections. I used them here in this chapter.Disclaimer: I don"t own Ayashi no Ceres. Only Yuu Watase owns it. How many times do I need to say that --Chapter 2 Ceres appears Since ancient times, people have adored the beauty of the sunset. The perfectly shaped orange sun was like a king with the red orange sky as his cape. Nothing could compare the feeling of calmness it brought about. It was just so perfect. It was something that could take away all your negative feelings such as sadness and emptiness. Who would have thought that in this occasion its power had no effect? In the gloomy chapel lit by those spooky candles, no one seemed to care about the sunset. No one bothered to look out in the window to experience the warmth the sunset held, to let it cheer him up and take his worries away. But of course, with the very heavy atmos

We see quite some opening/ending notes in the mistakes, which is interesting

| i | true → pred | margin | what the snippet is |
|---|---|---|---|
| 95 | D → I | 0.04 | Konichiwa! Firs of all, arigatou gozaimasu for all those who reviewed! You inspired me a lot.|
| 141 | D → P | 0.76 | This would not have been possible without all of you! Finally, the Epilogue :D Enjoy!  |
| 166 | B → N | 1.37 | "I chose the Spanish version because it"s a little different from the English and just fits better into the story." |
| 179 | K → P | 6.21 | "Wow, I really didn"t expect people to actually like this story! So, I have decided to continue it!"|

## Overlap with train, dev and test

In [7]:
def ngrams(text, n=32):
    words = re.findall(r"\w+", text.lower())
    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}


train_ngrams = set().union(*(ngrams(t) for t in train_texts))

for split in ["dev", "test"]:
    overlap = [len(ngrams(t) & train_ngrams) / len(ngrams(t)) for t in splits[split][0]]
    print(split, "snippets with more than 50% overlap:", round(np.mean(np.array(overlap) > 0.5), 2))

dev snippets with more than 50% overlap: 0.44
test snippets with more than 50% overlap: 0.46


While reading for failure analysis, many things felt similar, this combined with the really great results of our classifier made me a bit dubious. pon inspection we see that 46% of all test snippets share more than 50% overlap with the training data. This is concerning, as we then do not have a clean test or dev set.